# Methodology Walkthrough

This notebook walks through the complete pipeline step-by-step with explanations.

## What you'll see:
1. How Bright Data's AI Mode Scraper is called
2. The structure of the raw response (citations, #:~:text= URLs)
3. Text fragment decoding in action
4. Source page position finding
5. A live mini-analysis on a small sample

In [ ]:
# Setup
import json
import os
from pathlib import Path
from urllib.parse import unquote

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

RAW_DIR = Path('../data/raw')
PARSED_DIR = Path('../data/parsed')

print('Environment loaded.')
print(f'API key present: {bool(os.environ.get("BRIGHTDATA_API_KEY"))}')

## Step 1: Bright Data API Call Structure

Each query is submitted as a JSON payload to Bright Data's trigger endpoint.

In [ ]:
# Show the exact payload structure
example_payload = [
    {
        "url": "https://www.google.com/search?udm=50",
        "prompt": "what are the symptoms of type 2 diabetes",
        "country": "US"
    }
]

print('Trigger endpoint:')
print('  POST https://api.brightdata.com/datasets/v3/trigger?dataset_id=gd_mcswdt6z2elth3zqr2')
print()
print('Payload:')
print(json.dumps(example_payload, indent=2))

## Step 2: Understanding the Response Structure

The scraper returns structured data with the answer text and a `citations` array.
The key field is `citations[].url` — this contains the `#:~:text=` fragment.

In [ ]:
# Show response structure (using a synthetic example)
example_response = {
    "prompt": "what are the symptoms of type 2 diabetes",
    "answer_text": "Type 2 diabetes symptoms include increased thirst, frequent urination, fatigue, and blurred vision.",
    "citations": [
        {
            "url": "https://www.mayoclinic.org/diseases-conditions/type-2-diabetes/symptoms-causes/syc-20351193#:~:text=Type%202%20diabetes%20symptoms%20often%20develop%20slowly",
            "domain": "mayoclinic.org",
            "title": "Type 2 diabetes - Symptoms and causes",
            "description": "Type 2 diabetes — Comprehensive overview covers symptoms, treatment, prevention...",
            "cited": True
        },
        {
            "url": "https://www.cdc.gov/diabetes/basics/symptoms.html#:~:text=You%20might%20notice%20that%20you%20are%20thirsty",
            "domain": "cdc.gov",
            "title": "Diabetes Symptoms",
            "cited": True
        }
    ]
}

print('Response structure:')
print(json.dumps(example_response, indent=2))

## Step 3: Decoding the `#:~:text=` Fragment

This is the core operation that makes sentence-level analysis possible.

In [ ]:
def decode_text_fragment(url: str) -> dict:
    """Decode the #:~:text= fragment from a citation URL."""
    if '#:~:text=' not in url:
        return {'has_fragment': False, 'cited_sentence': None}
    
    fragment = url.split('#:~:text=', 1)[1]
    fragment = fragment.split('&')[0]
    decoded = unquote(fragment.replace('+', ' '))
    
    # Parse: [prefix-,]textStart[,textEnd][,-suffix]
    parts = decoded.split(',')
    text_parts = []
    for part in parts:
        if not part.endswith('-') and not part.startswith('-'):
            text_parts.append(part.strip())
    
    return {
        'has_fragment': True,
        'fragment_raw': decoded,
        'cited_sentence': text_parts[0] if text_parts else decoded,
    }

# Test it
for cite in example_response['citations']:
    result = decode_text_fragment(cite['url'])
    print(f"Domain: {cite['domain']}")
    print(f"Cited sentence: '{result['cited_sentence']}'")
    print()

## Step 4: Load Actual Parsed Data (after running scripts 01-03)

In [ ]:
# Load citations.csv if available
cite_path = PARSED_DIR / 'citations.csv'

if cite_path.exists():
    cite_df = pd.read_csv(cite_path)
    print(f'Loaded {len(cite_df)} citations')
    print(f'Columns: {list(cite_df.columns)}')
    print()
    print('Sample:')
    display(cite_df[['platform', 'query', 'domain', 'cited_sentence', 'cited_sentence_word_count', 'has_text_fragment']].head(10))
else:
    print('citations.csv not found — run scripts 01-03 first.')
    print('Expected path:', cite_path)

In [ ]:
# Fragment coverage stats
if cite_path.exists():
    print('Fragment Coverage:')
    print(f"  Total citations: {len(cite_df)}")
    print(f"  With #:~:text=: {cite_df['has_text_fragment'].sum()} ({cite_df['has_text_fragment'].mean():.1%})")
    print(f"  Without: {(~cite_df['has_text_fragment']).sum()}")
    print()
    print('By platform:')
    print(cite_df.groupby('platform')['has_text_fragment'].agg(['sum', 'mean', 'count']))